In [ ]:
#importation des modules
import torch
from torchvision.models import ResNet18_Weights
import torchvision.models as models
import numpy as np
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.utils.data import random_split    
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import Subset
from torch.utils.data import WeightedRandomSampler
from collections import Counter
import os
import torch.nn as nn
import gc
from sklearn.metrics import roc_curve, auc
from torch.cuda.amp import GradScaler, autocast


In [ ]:
#on récupère le dataset miniDDSM restructuré et traité depuis le s3
!mc mirror s3/lucasvital/stat_app/nouveau_dataset_mini_ddsm_700_700/ /home/onyxia/data/MINI-DDSM/nouveau_dataset_mini_ddsm_700_700
chemin_data="/home/onyxia/data/MINI-DDSM/nouveau_dataset_mini_ddsm_700_700"

In [ ]:
#importation du dataset vindr et miniddsm
chemin_data_vindr="/home/onyxia/work/dataset_vindr" #à définir
chemin_data_miniddsm="/home/onyxia/data/MINI-DDSM/nouveau_dataset_mini_ddsm_700_700"

os.makedirs(chemin_data_vindr,exist_ok=True)
os.makedirs(os.path.join(chemin_data_vindr,"0-normal"),exist_ok=True)
os.makedirs(os.path.join(chemin_data_vindr,"1-cancer_benign"),exist_ok=True)

!mc mirror s3/lucasvital/stat_app/0-normal/ /home/onyxia/work/dataset_vindr/0-normal
!mc mirror s3/lucasvital/stat_app/1-cancer_benign/ /home/onyxia/work/dataset_vindr/1-cancer_benign


!mc mirror s3/lucasvital/stat_app/nouveau_dataset_mini_ddsm_700_700/ /home/onyxia/data/MINI-DDSM/nouveau_dataset_mini_ddsm_700_700

In [ ]:
# Génération des données équilibrées pour Vindr (standardisation robuste par image)
#choix des paramètres de génération des données (à choisir identique au modèle entraîné)
resolution=672 #resolution des images transformées
bs = 32 #batch size 

#choix de normalisation des images
choix_normalisation_image = 0 #à choisir entre 0- pas de normalisation, 1-normalisation_par_image 2-normalisation_sur_le_dataset 
limite_basse=0.05 #à choisir entre proche de 0 (on exclut le fond noir de la normalisation et on le fixe arbitrairement bas ensuite)




def generation_donnes_eval(chemin_data,mode): #mode = 1 si on veut évaluer le dataset, mode = 0 si on veut juste calculer les quantiles et la distribution 

    #définition de la fonction de pour normaliser les images par rapport à elles mêmes
    def robust_standardize1(x):
        mask = x > limite_basse         #On crée un masque pour ignorer le fond (souvent proche de 0 ou < 0.05)
        if mask.any():
            mean = x[mask].mean()
            std = x[mask].std()
            x[mask] = (x[mask] - mean) / (std + 1e-6) 
            x[mask] = torch.clamp(x[mask], -3, 3) #on ramène les valeurs extrêmes du sein à l'intervalle -3 3
        x[~mask] = -4.0 # on fixe le reste des valeurs correspondantes au support noir à -4
        return x


    #definition de la fonction pour normaliser les images par rapport au dataset MiniDDSM et calcul de mean et std error du dataset
    def calcul_dataset_mean_std():

        def recuperation_pixel(loader_source, n_batches=20):
            # On récupère quelques données pour avoir une distribution stable
            source_pixels = []
            
            with torch.no_grad():
                # Extraction du domaine Source
                for i, (images, _) in tqdm(enumerate(loader_source)):
                    if i >= n_batches: break
                    # On prend un seul canal (gris) et on aplatit
                    source_pixels.extend(images[:, 0, :, :].flatten().numpy())
            return(source_pixels)

        def robust_mean_variance(x):
            # 1. On calcule la std error et la mean sur le dataset
            mask = (x > limite_basse)  
            if mask.any():
                mean = x[mask].mean()
                std = x[mask].std()
            return (mean,std)

        train_transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor()])

        train_dataset_full = ImageFolder(root=chemin_data, transform=train_transform)
        train_loader1 = DataLoader(train_dataset_full, batch_size=bs)

        (source_pixel)=recuperation_pixel(train_loader1)
        source_pixel=np.array(source_pixel)

        (mu,sigma)=robust_mean_variance(source_pixel)
        return(mu,sigma)

    def robust_standardize2(x):
        mask = x > limite_basse
        if mask.any():
            mean = mu
            std = sigma
            x[mask] = (x[mask] - mean) / (std + 1e-6)    
            x[mask] = torch.clamp(x[mask], -3, 3)
        x[~mask] = -4.0 
        return x

    if choix_normalisation_image==0:
        def identity(x):
            return x
        fonction = identity
    elif choix_normalisation_image==1:
        fonction=robust_standardize1
    else:
        (mu,sigma)=calcul_dataset_mean_std()
        fonction=robust_standardize2
    
    if mode == 0:
        transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor(),
        transforms.Lambda(fonction)])

    elif mode == 1:
        transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor(),
        transforms.Lambda(fonction), 
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    data_set=ImageFolder(root=chemin_data, transform=transform)

    label=np.array(data_set.targets)
    print(f"nombre d'images : {len(label)}")
    print(f"proportion d'image benign-cancer : {label.sum()/len(label)}")

    test_loader = DataLoader(
        data_set, 
        batch_size=bs,       # Augmente le batch_size si possible (ex: 64 ou 128)
        shuffle=False, 
        num_workers=4,       # <--- ESSENTIEL : utilise plusieurs cœurs CPU
        pin_memory=True      # <--- Accélère le transfert CPU vers GPU
        )
    return(test_loader)

In [ ]:
# Génération des données équilibrées (standardisation robuste par image) pour DDSM
#choix des paramètres de génération des données et de l'entraînement
nom_modele_sauvegarde="best_model_672_EWC" #nom du modèle sauvegardé
#ATTENTION : IL FAUT MODIFIER LE NOM DE SAUVEGARDE DANS LE S3 DANS LE CODE DENTRAINEMENT SINON LE MODELE N EST PAS ENREGISTRE DANS LE S3 !!!
resolution=672 #resolution des images transformées
bs = 32 #batch size 

#paramètres de data augmentation / rajout de bruit dans l'entraînement
contrast=0 #on transforme de +/- x<10% le contraste de l'image (selon une loi uniforme)
brightness=0 #on transforme de +/- x< 10% la luminosité de l'image (selon une loi uniforme)
degrees=0 #on fait une rotation de +/- x<15% de l'image (selon une loi uniforme) NB: les bords nouvellement crées sont noirs

#choix de normalisation des images
choix_normalisation_image = 0 #à choisir entre 0- pas de normalisation, 1-normalisation_par_image 2-normalisation_sur_le_dataset 
limite_basse=0.05 #à choisir entre proche de 0 (on exclut le fond noir de la normalisation et on le fixe arbitrairement bas ensuite)

# Géneration de donnes équilibrées
numero_random = 42 #on choisit l'aléatoire pour la génération d'un dataset d'entraînement 
train_size = 0.7 #on choisit la taille du dataset d'entraînement
val_size = 0.15 #on choisit la taille du dataset de validation
test_size = 0.15 #on choisit la taille du dataset de test
#methode_generation = surepresentation #à choisir entre sureprésenter les images de cancer ou sous reprenter les images normales

#paramètres d'entraînement
dropout = 0.5 # Désactive 50% des neurones aléatoirement à chaque itération pour éviter l'overfitting 
lr=0.0001 #learning rate
weight_decay=1e-4 #poids de pénalisation de surapprentissage lorsqu'un paramètre devient trop grand
factor=0.1 #facteur d'actualisation du learning rate par le scheduler
patience=5 #patience du scheduler
threshold=0.0001 #seuil du scheduler
patience_limite = 12 #patience limite (l'entraînement s'arrête si il n'y a pas d'amélioration de la Loss de Validation au bout de 12 epochs)
num_epochs = 80  #nombre d'épochs maximum 



def generation_donnes():

    #définition de la fonction de pour normaliser les images par rapport à elles mêmes
    def robust_standardize1(x):
        mask = x > limite_basse         #On crée un masque pour ignorer le fond (souvent proche de 0 ou < 0.05)
        if mask.any():
            mean = x[mask].mean()
            std = x[mask].std()
            x[mask] = (x[mask] - mean) / (std + 1e-6) 
            x[mask] = torch.clamp(x[mask], -3, 3) #on ramène les valeurs extrêmes du sein à l'intervalle -3 3
        x[~mask] = -4.0 # on fixe le reste des valeurs correspondantes au support noir à -4
        return x


    #definition de la fonction pour normaliser les images par rapport au dataset MiniDDSM et calcul de mean et std error du dataset
    def calcul_dataset_mean_std():

        def recuperation_pixel(loader_source, n_batches=20):
            # On récupère quelques données pour avoir une distribution stable
            source_pixels = []
            
            with torch.no_grad():
                # Extraction du domaine Source
                for i, (images, _) in tqdm(enumerate(loader_source)):
                    if i >= n_batches: break
                    # On prend un seul canal (gris) et on aplatit
                    source_pixels.extend(images[:, 0, :, :].flatten().numpy())
            return(source_pixels)

        def robust_mean_variance(x):
            # 1. On calcule la std error et la mean sur le dataset
            mask = (x > 0.05) & (x<0.97) 
            if mask.any():
                mean = x[mask].mean()
                std = x[mask].std()
            return (mean,std)

        train_transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor()])

        train_dataset_full = ImageFolder(root=chemin_data, transform=train_transform)
        train_loader1 = DataLoader(train_dataset_full, batch_size=bs)

        (source_pixel)=recuperation_pixel(train_loader1)
        source_pixel=np.array(source_pixel)

        (mu,sigma)=robust_mean_variance(source_pixel)
        return(mu,sigma)

    def robust_standardize2(x):
        mask = x > limite_basse
        if mask.any():
            mean = mu
            std = sigma
            x[mask] = (x[mask] - mean) / (std + 1e-6)    
            x[mask] = torch.clamp(x[mask], -3, 3)
        x[~mask] = -4.0 
        return x

    if choix_normalisation_image==0:
        def identity(x):
            return x
        fonction = identity
    elif choix_normalisation_image==1:
        fonction=robust_standardize1
    else:
        (mu,sigma)=calcul_dataset_mean_std()
        fonction=robust_standardize2
        
    # Transformation des données ATTENTION A L'ORDRE DES TRANSFORMS!!!
    train_transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ColorJitter(brightness=brightness, contrast=contrast), # Change légèrement la luminosité
        transforms.RandomRotation(degrees=degrees),       # Rotation légère (en mettant du noir 0 sur le côte)
        transforms.RandomVerticalFlip(p=0.5),        # Les mammo peuvent être inversées
        transforms.ToTensor(),
        transforms.Lambda(fonction), 
        # --- AJOUTS POUR LUTTER CONTRE L'OVERFITTING ---
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor(),
        transforms.Lambda(fonction), 
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    train_dataset_full = ImageFolder(root=chemin_data, transform=train_transform)
    val_dataset_full = ImageFolder(root=chemin_data, transform=transform)

    label = np.array(val_dataset_full.targets)
    print(f"Nombre total d'images : {len(label)}")
    print(f"Classes détectées : {val_dataset_full.classes}")
    print(f"Distribution des classes : {Counter(label)}")
    print(f"accuracy random val : {1-(label.sum()/len(label))}")

    #on équilibre la proportion de labels 0 et 1 dans le dataset d'entraînement
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=numero_random)
    for train_val_index, test_index in sss1.split(np.zeros(len(label)), label):
        train_val_indices = train_val_index
        test_indices = test_index

    train_val_labels = label[train_val_indices]
    new_val_size = val_size / (train_size + val_size)
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=new_val_size, random_state=numero_random)
    for train_index, val_index in sss2.split(np.zeros(len(train_val_labels)), train_val_labels):
        train_indices = train_val_indices[train_index]
        val_indices = train_val_indices[val_index]

    print(f"Train set: {len(train_indices)}")
    print(f"Validation set: {len(val_indices)}")
    print(f"Test set: {len(test_indices)}")

    # Weighted Sampler pour équilibrer les classes
    train_labels = label[train_indices]
    class_counts = Counter(train_labels)
    print(f"Distribution dans train : {class_counts}")

    num_labels = len(train_labels)
    weight_for_class = {cl: num_labels / count for cl, count in class_counts.items()}
    sample_weights = [weight_for_class[l] for l in train_labels]
    sample_weights_tensor = torch.DoubleTensor(sample_weights)

    print(f"Poids par classe : {weight_for_class}")

    sampler = WeightedRandomSampler(
        weights=sample_weights_tensor, 
        num_samples=len(sample_weights_tensor), 
        replacement=True
    )

    train_set = Subset(train_dataset_full, train_indices)
    val_set = Subset(val_dataset_full, val_indices)
    test_set = Subset(val_dataset_full, test_indices)

    train_loader = DataLoader(train_set, batch_size=bs, num_workers=4, pin_memory=True, sampler=sampler)
    val_loader = DataLoader(val_set, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    return(train_loader,val_loader,test_loader)

(train_loader_ddsm,val_loader_ddsm,test_loader_ddsm)=generation_donnes()

In [ ]:
#choix d'importation du modèle
!mc cp s3/lucasvital/stat_app/new2_best_model_672.pth /home/onyxia/work/best_resnet18_model.pth

In [ ]:
#calcul de l'information de Fischer
#récupération du dataset Vindr transformé pour l'évaluer
test_loader_vindr=generation_donnes_eval(chemin_data=chemin_data_vindr,mode=1)

#récupération du modèle
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
num_class=2
model.fc = nn.Sequential(
    nn.Dropout(p=0.5), # Désactive 50% des neurones aléatoirement à chaque itération
    nn.Linear(model.fc.in_features, num_class))

model.load_state_dict(torch.load("/home/onyxia/work/best_resnet18_model.pth"))

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"GPU détecté : {torch.cuda.get_device_name(0)}")



def compute_fisher(model, dataloader, device):
    fisher = {}
    params = {n: p for n, p in model.named_parameters() if p.requires_grad}
    
    # Initialiser Fisher à zéro
    for n, p in params.items():
        fisher[n] = torch.zeros_like(p.data)

    model.eval()
    for images, labels in dataloader:
        images = images.to(device)
        model.zero_grad()
        
        outputs = model(images)
        # On calcule la log-vraisemblance (BCE)
        prob = torch.sigmoid(outputs)
        # On utilise les prédictions du modèle pour calculer Fisher (selon l'article)
        log_likelihood = torch.nn.functional.binary_cross_entropy_with_logits(outputs, prob, reduction='sum')
        
        log_likelihood.backward()

        for n, p in params.items():
            if p.grad is not None:
                fisher[n] += (p.grad ** 2) / len(dataloader.dataset)
                
    return fisher

# Juste avant de passer à VinDr :
opt_weights = {}
for name, param in model.named_parameters():
    # .clone() est crucial pour avoir une copie indépendante
    # .detach() assure que ces poids ne seront pas modifiés par le gradient
    opt_weights[name] = param.data.clone().detach()

#pour le calul de la loss
def ewc_loss(model, current_loss, fisher, opt_weights, lam=400):
    ewc_penalty = 0
    for n, p in model.named_parameters():
        if n in fisher:
            # Le "ressort" : importance * (poids_actuel - poids_DDSM)^2
            ewc_penalty += (fisher[n] * (p - opt_weights[n])**2).sum()
    
    return current_loss + (lam / 2) * ewc_penalty

In [ ]:
# Entraînement du modèle
# 1. Calcule Fisher sur DDSM (avec le code compute_fisher vu précédemment)
fisher_matrix = compute_fisher(model, train_loader_ddsm, device)

# 3. Définit l'importance du ressort (Hyperparamètre lambda)
importance = 1000


criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay) #ajout d'un weightdecay pour pénaliser l'overfitting
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=factor, patience=patience, threshold=threshold)


# Fonctions d'entraînement et validation
def train_model(model, train_loader, val_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    scaler = GradScaler() # Le scaler reste identique
    
    for inputs, labels in tqdm(train_loader, desc="Entraînement"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
    
        with autocast(): 
            outputs = model(inputs)
            current_loss = criterion(outputs, labels)
            loss=ewc_loss(model=model, current_loss=current_loss, fisher=fisher_matrix, opt_weights=opt_weights, lam=importance)
        
        # 4. On modifie la phase de rétropropagation
        scaler.scale(loss).backward()  # On "scale" la loss pour pas qu'elle soit trop petite
        scaler.step(optimizer)         # L'optimizer fait son pas
        scaler.update()                # On met à jour le scaler pour le prochain tour
        
        running_loss += loss.item() * inputs.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    return epoch_loss


def validate_model(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    corrects = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
    
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_acc = corrects.double() / len(val_loader.dataset)
    
    return epoch_loss, epoch_acc

# Boucle d'entraînement avec early stopping
best_acc = 0.0 #initialisation accuracy
patience_compteur = 0 #initialisation patience
best_loss=10 #initialisation Loss

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Époque {epoch+1}/{num_epochs}")
    print(f"{'='*50}")
    
    # Entraînement
    train_loss = train_model(model, train_loader, val_loader, criterion, optimizer)
    print(f"Loss d'entraînement: {train_loss:.4f}")
    
    # Validation
    val_loss, val_acc = validate_model(model, val_loader, criterion, device)
    print(f"Loss de validation: {val_loss:.4f} | Accuracy: {val_acc:.4f}")
    
    # Scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning rate: {current_lr}")
    
    # Sauvegarde du meilleur modèle AValidation Loss
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "/home/onyxia/"+nom_modele_sauvegarde+"loss.pth")
        # Sauvegarder sur S3
        !mc cp /home/onyxia/new_best_model_700_EWC.pth s3/lucasvital/stat_app/
        patience_compteur = 0
    else:
        patience_compteur += 1
        print(f"Patience: {patience_compteur}/{patience_limite}")

    if patience_compteur >= patience_limite:
        print("\n Early stopping déclenché")
        break

print(f"\n{'='*50}")
print(f"Entraînement terminé. Meilleure accuracy: {best_acc:.4f}")
print(f"{'='*50}")